## LCEL 인터페이스


사용자 정의 체인을 가능한 쉽게 만들 수 있도록, [`Runnable`](https://api.python.langchain.com/en/stable/runnables/langchain_core.runnables.base.Runnable.html#langchain_core.runnables.base.Runnable) 프로토콜을 구현했습니다. 

`Runnable` 프로토콜은 대부분의 컴포넌트에 구현되어 있습니다.

이는 표준 인터페이스로, 사용자 정의 체인을 정의하고 표준 방식으로 호출하는 것을 쉽게 만듭니다.
표준 인터페이스에는 다음이 포함됩니다.

- [`stream`](#stream): 응답의 청크를 스트리밍합니다.
- [`invoke`](#invoke): 입력에 대해 체인을 호출합니다.
- [`batch`](#batch): 입력 목록에 대해 체인을 호출합니다.

비동기 메소드도 있습니다.

- [`astream`](#async-stream): 비동기적으로 응답의 청크를 스트리밍합니다.
- [`ainvoke`](#async-invoke): 비동기적으로 입력에 대해 체인을 호출합니다.
- [`abatch`](#async-batch): 비동기적으로 입력 목록에 대해 체인을 호출합니다.
- [`astream_log`](#async-stream-intermediate-steps): 최종 응답뿐만 아니라 발생하는 중간 단계를 스트리밍합니다.

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


LCEL 문법을 사용하여 chain 을 생성합니다.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatOpenAI 모델을 인스턴스화합니다.
model = ChatOpenAI()
# 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")
# 프롬프트와 모델을 연결하여 대화 체인을 생성합니다.
chain = prompt | model | StrOutputParser()

In [4]:
chain

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic} 에 대하여 3문장으로 설명해줘.')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001A970B3B550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001A971056B50>, root_client=<openai.OpenAI object at 0x000001A970B3B0D0>, root_async_client=<openai.AsyncOpenAI object at 0x000001A971056850>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser()

## stream: 실시간 출력


이 함수는 `chain.stream` 메서드를 사용하여 주어진 토픽에 대한 데이터 스트림을 생성하고, 이 스트림을 반복하여 각 데이터의 내용(`content`)을 즉시 출력합니다. `end=""` 인자는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True` 인자는 출력 버퍼를 즉시 비우도록 합니다. 

In [5]:
# chain.stream 메서드를 사용하여 '멀티모달' 토픽에 대한 스트림을 생성하고 반복합니다.
for token in chain.stream({"topic": "멀티모달"}):
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

c:\Users\heesu\AppData\Local\pypoetry\Cache\virtualenvs\langchain-kr-IlBIRnxo-py3.11\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


멀티모달은 여러 가지 다른 형식의 매체를 사용하여 정보를 전달하거나 상호작용하는 것을 의미합니다. 이는 텍스트, 이미지, 음성, 비디오 등 다양한 형식을 포함할 수 있습니다. 멀티모달은 사용자에게 더 풍부한 경험을 제공하고, 정보를 더욱 효율적으로 전달할 수 있도록 도와줍니다.

## invoke: 호출


`chain` 객체의 `invoke` 메서드는 주제를 인자로 받아 해당 주제에 대한 처리를 수행합니다.

In [13]:
# chain 객체의 invoke 메서드를 호출하고, 'ChatGPT'라는 주제로 딕셔너리를 전달합니다.
chain.invoke({"topic": "ChatGPT"})

'ChatGPT는 인공지능 대화 시스템으로, 자연어 처리 기술을 이용하여 사람과의 대화를 모방합니다. 사용자의 질문에 대답하거나 대화를 이어가는 역할을 하며, 다양한 주제에 대해 정보를 제공합니다. 학습을 통해 계속 발전하고, 사용자들에게 보다 유익한 대화 경험을 제공하고 있습니다.'

## batch: 배치(단위 실행)


함수 `chain.batch`는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 `topic` 키의 값을 사용하여 일괄 처리를 수행합니다.

In [10]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
# chain.batch([{"topic": "ChatGPT"}, {"topic": "Instagram"}])
answer = chain.batch([{"topic": "ChatGPT"}, {"topic": "Instagram"}])

In [11]:
answer[0]

'ChatGPT는 사용자와 대화하는 자연어 처리 모델로, 자연스러운 대화를 제공합니다. 감정, 지식, 상황에 맞게 대화할 수 있어 다양한 주제에 대해 이야기를 나눌 수 있습니다. 인공지능 기술을 활용해 대화할 때 사용자의 니즈를 충족시키고 효율적인 정보 전달이 가능합니다.'

In [12]:
answer[1]

'Instagram은 사진과 동영상을 공유할 수 있는 소셜 미디어 플랫폼이다. 다양한 필터와 편집 기능을 통해 사용자들이 자신의 콘텐츠를 보다 멋지게 표현할 수 있다. 또한 팔로워와 소통을 통해 유명인이나 브랜드와 가까이서 소통할 수 있는 장점이 있다.'

`max_concurrency` 매개변수를 사용하여 동시 요청 수를 설정할 수 있습니다

`config` 딕셔너리는 `max_concurrency` 키를 통해 동시에 처리할 수 있는 최대 작업 수를 설정합니다. 여기서는 최대 3개의 작업을 동시에 처리하도록 설정되어 있습니다.

In [14]:
chain.batch(
    [
        {"topic": "ChatGPT"},
        {"topic": "Instagram"},
        {"topic": "멀티모달"},
        {"topic": "프로그래밍"},
        {"topic": "머신러닝"},
    ],
    config={"max_concurrency": 3},
)

['ChatGPT는 AI 기술을 활용하여 대화형 상호작용을 가능하게 하는 자연어 처리 모델이다. 사용자는 텍스트 기반으로 ChatGPT와 대화를 나눌 수 있으며, 문맥을 이해하고 적절한 응답을 생성한다. 다양한 분야에서 사용되며, 고객 서비스, 교육, 엔터테인먼트 등 다양한 분야에서 활용되고 있다.',
 'Instagram은 사진과 동영상을 공유할 수 있는 소셜 미디어 플랫폼이다. 사용자들은 다양한 필터와 편집 기능을 통해 자신만의 콘텐츠를 만들어 공유할 수 있다. 또한 팔로워들과 소통하며 다른 사용자들의 콘텐츠를 감상할 수 있는 인기 있는 앱이다.',
 '멀티모달은 여러 가지 다양한 형식의 정보를 결합하여 제공하는 시스템이다. 이는 텍스트, 이미지, 음성, 동영상 등 다양한 매체를 활용하여 정보를 전달하고 상호작용하는 방식이다. 멀티모달은 사용자들에게 보다 풍부하고 효과적인 정보 전달을 가능하게 하며, 다양한 인지 수준에 맞게 정보를 이해하고 활용할 수 있다.',
 '프로그래밍은 컴퓨터에게 작업을 시키기 위해 일련의 명령을 작성하는 과정이다. 이러한 명령들은 프로그래밍 언어를 통해 작성되며, 문제 해결 및 원하는 결과물을 얻기 위해 사용된다. 프로그래머는 이러한 명령을 작성하여 컴퓨터가 원하는 동작을 수행하도록 한다.',
 '머신러닝은 컴퓨터가 데이터를 학습하고 패턴을 발견하여 예측을 수행하는 인공지능 기술이다. 이를 위해 알고리즘과 모델을 사용하여 데이터를 분석하고 학습시키는 프로세스를 반복한다. 머신러닝은 이미지 및 음성 인식, 자연어 처리 등 다양한 분야에 활용되고 있다.']

## async stream: 비동기 스트림


함수 `chain.astream`은 비동기 스트림을 생성하며, 주어진 토픽에 대한 메시지를 비동기적으로 처리합니다.

비동기 for 루프(`async for`)를 사용하여 스트림에서 메시지를 순차적으로 받아오고, `print` 함수를 통해 메시지의 내용(`s.content`)을 즉시 출력합니다. `end=""`는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True`는 출력 버퍼를 강제로 비워 즉시 출력되도록 합니다.


In [15]:
# 비동기 스트림을 사용하여 'YouTube' 토픽의 메시지를 처리합니다.
async for token in chain.astream({"topic": "YouTube"}):
    # 메시지 내용을 출력합니다. 줄바꿈 없이 바로 출력하고 버퍼를 비웁니다.
    print(token, end="", flush=True)

YouTube는 온라인 동영상 공유 플랫폼으로 영상을 업로드하고 시청할 수 있는 서비스이다. 다양한 주제의 동영상 콘텐츠를 제공하며 사용자들은 무료로 시청할 수 있다. 또한 유저들은 구독을 통해 좋아하는 채널을 팔로우하고 새로운 동영상을 받을 수 있다.

## async invoke: 비동기 호출


`chain` 객체의 `ainvoke` 메서드는 비동기적으로 주어진 인자를 사용하여 작업을 수행합니다. 여기서는 `topic`이라는 키와 `NVDA`(엔비디아의 티커) 라는 값을 가진 딕셔너리를 인자로 전달하고 있습니다. 이 메서드는 특정 토픽에 대한 처리를 비동기적으로 요청하는 데 사용될 수 있습니다.


In [16]:
# 비동기 체인 객체의 'ainvoke' 메서드를 호출하여 'NVDA' 토픽을 처리합니다.
my_process = chain.ainvoke({"topic": "NVDA"})

In [18]:
my_process

<coroutine object RunnableSequence.ainvoke at 0x000001A970F42C40>

In [17]:
# 비동기로 처리되는 프로세스가 완료될 때까지 기다립니다.
await my_process

'NVDA는 전 세계적으로 널리 사용되는 시각장애인을 위한 화면 낭독 소프트웨어이다. 이 소프트웨어는 컴퓨터에 텍스트를 읽어주고 사용자가 키보드나 마우스를 조작할 수 있도록 도와준다. NVDA는 무료로 제공되며 오픈 소스로 개발되어 사용자들이 소프트웨어를 수정하고 개선할 수 있다.'

## async batch: 비동기 배치


함수 `abatch`는 비동기적으로 일련의 작업을 일괄 처리합니다.

이 예시에서는 `chain` 객체의 `abatch` 메서드를 사용하여 `topic` 에 대한 작업을 비동기적으로 처리하고 있습니다.

`await` 키워드는 해당 비동기 작업이 완료될 때까지 기다리는 데 사용됩니다.


In [19]:
# 주어진 토픽에 대해 비동기적으로 일괄 처리를 수행합니다.
my_abatch_process = chain.abatch(
    [{"topic": "YouTube"}, {"topic": "Instagram"}, {"topic": "Facebook"}]
)

In [20]:
# 비동기로 처리되는 일괄 처리 프로세스가 완료될 때까지 기다립니다.
await my_abatch_process

['YouTube는 구글이 소유하고 있는 세계적인 온라인 동영상 플랫폼으로, 사용자들은 무료로 다양한 내용을 시청하고 업로드할 수 있다. 다양한 채널과 영상들이 있어 사용자들이 원하는 콘텐츠를 쉽게 찾을 수 있으며, 댓글이나 좋아요 등의 기능을 통해 상호 소통이 가능하다. 광고 수익을 얻을 수 있는 유튜버들도 많이 활동하고 있어 미디어 산업의 중심지로 자리매김했다.',
 'Instagram은 사진과 동영상을 공유하는 소셜 미디어 플랫폼으로, 사용자들은 자신의 일상을 다른 사람들과 공유하고 소통할 수 있다.\n해시태그를 활용하여 관심사나 주제에 맞는 컨텐츠를 발견할 수 있고, 팔로워들과의 상호작용을 통해 커뮤니케이션을 할 수 있다.\n인스타그램 스토리와 라이브 기능을 통해 실시간으로 소통할 수 있으며, 다양한 필터와 편집 기능을 통해 창의적인 콘텐츠를 제공할 수 있다.',
 'Facebook은 전 세계에서 가장 인기 있는 소셜 네트워킹 서비스 중 하나로 사용자들이 다양한 콘텐츠를 공유하고 소통할 수 있는 플랫폼이다. 사용자들은 친구와 가족과 소통하고 새로운 사람들을 만날 수 있으며 비즈니스나 기관들도 마케팅 및 홍보 활동을 할 수 있다. 또한 개인 정보 보호와 악성 콘텐츠로부터의 보호를 위한 다양한 보안 기능도 제공된다.']

## Parallel: 병렬성

LangChain Expression Language가 병렬 요청을 지원하는 방법을 살펴봅시다.
예를 들어, `RunnableParallel`을 사용할 때, 각 요소를 병렬로 실행합니다.


`langchain_core.runnables` 모듈의 `RunnableParallel` 클래스를 사용하여 두 가지 작업을 병렬로 실행하는 예시를 보여줍니다.

`ChatPromptTemplate.from_template` 메서드를 사용하여 주어진 `country`에 대한 **수도** 와 **면적** 을 구하는 두 개의 체인(`chain1`, `chain2`)을 만듭니다.

이 체인들은 각각 `model`과 파이프(`|`) 연산자를 통해 연결됩니다. 마지막으로, `RunnableParallel` 클래스를 사용하여 이 두 체인을 `capital`와 `area`이라는 키로 결합하여 동시에 실행할 수 있는 `combined` 객체를 생성합니다.


In [21]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

`chain1.invoke()` 함수는 `chain1` 객체의 `invoke` 메서드를 호출합니다.

이때, `country`이라는 키에 `대한민국`라는 값을 가진 딕셔너리를 인자로 전달합니다.


In [22]:
# chain1 를 실행합니다.
chain1.invoke({"country": "대한민국"})

'대한민국의 수도는 서울이야.'

이번에는 `chain2.invoke()` 를 호출합니다. `country` 키에 다른 국가인 `미국` 을 전달합니다.


In [23]:
# chain2 를 실행합니다.
chain2.invoke({"country": "미국"})

'미국의 총 면적은 대략 9,833,520 km² 입니다.'

`combined` 객체의 `invoke` 메서드는 주어진 `country`에 대한 처리를 수행합니다.

이 예제에서는 `대한민국`라는 주제를 `invoke` 메서드에 전달하여 실행합니다.


In [25]:
# 병렬 실행 체인을 실행합니다.
combined.invoke({"country": "대한민국"})

{'capital': '대한민국의 수도는 서울입니다.', 'area': '대한민국의 면적은 약 100,363km² 입니다.'}

변수가 다른 두 체인

In [26]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country1} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country2} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

In [27]:
# 병렬 실행 체인을 실행합니다.
combined.invoke({"country1": "대한민국", "country2": "대한민국"})

{'capital': '서울이야.', 'area': '대한민국의 총 면적은 약 100,210㎢ 입니다.'}

### 배치에서의 병렬 처리

병렬 처리는 다른 실행 가능한 코드와 결합될 수 있습니다.
배치와 병렬 처리를 사용해 보도록 합시다.


`chain1.batch` 함수는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 "topic" 키에 해당하는 값을 처리합니다. 이 예시에서는 "대한민국"와 "미국"라는 두 개의 토픽을 배치 처리하고 있습니다.


In [29]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

In [30]:
# 배치 처리를 수행합니다.
chain1.batch([{"country": "대한민국"}, {"country": "미국"}])

['대한민국의 수도는 서울입니다.', '미국의 수도는 워싱턴 D.C.입니다.']

`chain2.batch` 함수는 여러 개의 딕셔너리를 리스트 형태로 받아, 일괄 처리(batch)를 수행합니다.

이 예시에서는 `대한민국`와 `미국`라는 두 가지 국가에 대한 처리를 요청합니다.


In [31]:
# 배치 처리를 수행합니다.
chain2.batch([{"country": "대한민국"}, {"country": "미국"}])

['대한민국의 총 면적은 약 100,363제곱 킬로미터 입니다.', '미국의 면적은 약 9,826만 제곱 킬로미터 입니다.']

`combined.batch` 함수는 주어진 데이터를 배치로 처리하는 데 사용됩니다. 이 예시에서는 두 개의 딕셔너리 객체를 포함하는 리스트를 인자로 받아 각각 `대한민국`와 `미국` 두 나라에 대한 데이터를 배치 처리합니다.


In [32]:
# 주어진 데이터를 배치로 처리합니다.
combined.batch([{"country": "대한민국"}, {"country": "미국"}])

[{'capital': '대한민국의 수도는 서울입니다.', 'area': '대한민국의 총 면적은 약 100,363 km²입니다.'},
 {'capital': '미국의 수도는 워싱턴 D.C.입니다.',
  'area': '미국의 총 면적은 9,833,520km² 입니다. (3,796,742 제곱마일)'}]